# Notebook 03 — 3D Patch Extraction and Multi-Task Target Generation

This notebook extracts fixed-size 3D CT patches and their segmentation and classification targets from the persistent preprocessed RibFrac dataset produced by Notebook 02.

## Development strategy

Only **RibFrac128** is processed initially. Full split-wise extraction remains disabled until Notebooks 03–07 have been validated.

## Methodological principles

- Reuse the patient-level split created in Notebook 01/02; do not recreate it.
- Keep all patches from a patient in that patient's assigned split.
- Use identical coordinates for CT and annotation patches.
- Patch size: `(64, 64, 64)`; stride: `(32, 32, 32)`.
- Segmentation target: `0 = background`, `1 = valid fracture`.
- Classification target: `0 = background`, `1 = displaced`, `2 = non-displaced`, `3 = buckle`, `4 = segmental`.
- Map NIfTI annotation IDs to `label_code` using `ribfrac-train-info-1.csv`.
- Treat `label_code = -1` as ambiguous/ignored, never as background.
- A clean negative patch contains no valid fracture voxels and no ambiguous voxels.
- A positive patch contains valid fracture voxels and receives the dominant valid fracture class.
- A tie between multiple fracture classes is excluded because the present classifier is single-label softmax.
- No random patch-level train/validation split is performed.


In [ ]:
# Cell 1 — Import libraries
import gc
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

print("Libraries imported successfully.")


In [ ]:
# Cell 2 — Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
print("Random seed:", SEED)


In [ ]:
# Cell 3 — Define persistent input and working-output directories
DATASET_ROOT = Path("/kaggle/input/datasets/waboke/ribfrac-processed-dev")

PREPROCESSED_ROOT = DATASET_ROOT / "ribfrac_preprocessed"
PREPROCESSED_IMAGE_DIR = PREPROCESSED_ROOT / "images"
PREPROCESSED_LABEL_DIR = PREPROCESSED_ROOT / "labels"
PREPROCESSED_METADATA_DIR = PREPROCESSED_ROOT / "metadata"

INFO_PATH = DATASET_ROOT / "ribfrac_metadata" / "ribfrac-train-info-1.csv"

PATCH_ROOT = Path("/kaggle/working/patch_dataset")
PATCH_MANIFEST_ROOT = PATCH_ROOT / "manifests"
PATCH_ROOT.mkdir(parents=True, exist_ok=True)
PATCH_MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)

print("Dataset root:", DATASET_ROOT)
print("Preprocessed root:", PREPROCESSED_ROOT)
print("Annotation metadata:", INFO_PATH)
print("Patch output root:", PATCH_ROOT)


In [ ]:
# Cell 4 — Verify persistent dataset structure
required_paths = {
    "Dataset root": DATASET_ROOT,
    "Preprocessed root": PREPROCESSED_ROOT,
    "Image directory": PREPROCESSED_IMAGE_DIR,
    "Label directory": PREPROCESSED_LABEL_DIR,
    "Metadata directory": PREPROCESSED_METADATA_DIR,
    "Annotation metadata": INFO_PATH,
}

print("=" * 70)
print("VERIFYING PERSISTENT DATASET")
print("=" * 70)

for name, path in required_paths.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{name:<25}: {status}")
    print(f"  {path}")

if not all(path.exists() for path in required_paths.values()):
    raise FileNotFoundError("One or more required dataset paths are missing.")

print("=" * 70)
print("Persistent dataset verification: PASS")
print("=" * 70)


In [ ]:
# Cell 5 — Load annotation metadata and patient-level split
info_df = pd.read_csv(INFO_PATH)
SPLIT_PATH = PREPROCESSED_METADATA_DIR / "patient_split.csv"

if not SPLIT_PATH.exists():
    raise FileNotFoundError(f"Patient split not found: {SPLIT_PATH}")

split_df = pd.read_csv(SPLIT_PATH)

required_info_columns = {"public_id", "label_id", "label_code"}
required_split_columns = {"public_id", "split"}

if not required_info_columns.issubset(info_df.columns):
    raise ValueError(f"Annotation metadata must contain {required_info_columns}")
if not required_split_columns.issubset(split_df.columns):
    raise ValueError(f"Patient split must contain {required_split_columns}")

print("Annotation metadata shape:", info_df.shape)
print("Patient split shape:", split_df.shape)
print("
Patients per split:")
print(split_df["split"].value_counts())

train_ids = set(split_df.loc[split_df["split"] == "train", "public_id"])
val_ids = set(split_df.loc[split_df["split"] == "validation", "public_id"])
test_ids = set(split_df.loc[split_df["split"] == "test", "public_id"])

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

print("
Patient-level split integrity: PASS")


In [ ]:
# Cell 6 — Define patch-extraction parameters
PATCH_SIZE = (64, 64, 64)
STRIDE = (32, 32, 32)

NEGATIVE_TO_POSITIVE_RATIO = 1.0
MIN_FRACTURE_VOXELS = 1
AMBIGUOUS_LABEL_CODE = -1

CLASS_NAMES = {
    0: "background",
    1: "displaced",
    2: "non-displaced",
    3: "buckle",
    4: "segmental",
}

LABEL_CODE_TO_CLASS = {
    0: 0,
    1: 1,
    2: 2,
    3: 3,
    4: 4,
}

print("Patch size:", PATCH_SIZE)
print("Stride:", STRIDE)
print("Negative/positive ratio:", NEGATIVE_TO_POSITIVE_RATIO)
print("Classification classes:", CLASS_NAMES)


## Label handling

The label volume contains **annotation IDs**, not final fracture classes.

For each patient, the metadata provides `label_id -> label_code`.

- `0` = background
- `1` = displaced
- `2` = non-displaced
- `3` = buckle
- `4` = segmental
- `-1` = ambiguous/ignore

The extraction therefore creates separate valid-fracture and ambiguous masks. Ambiguous annotations are excluded rather than interpreted as background.


In [ ]:
# Cell 7 — Build annotation maps for one patient
def get_patient_annotation_maps(patient_id):
    patient_rows = info_df[info_df["public_id"] == patient_id].copy()
    if patient_rows.empty:
        raise ValueError(f"No annotation metadata found for {patient_id}")

    label_id_to_code = {}
    label_id_to_class = {}
    valid_label_ids = set()
    ambiguous_label_ids = set()

    for _, row in patient_rows.iterrows():
        label_id = int(row["label_id"])
        label_code = int(row["label_code"])
        label_id_to_code[label_id] = label_code

        if label_code in (1, 2, 3, 4):
            label_id_to_class[label_id] = LABEL_CODE_TO_CLASS[label_code]
            valid_label_ids.add(label_id)
        elif label_code == AMBIGUOUS_LABEL_CODE:
            ambiguous_label_ids.add(label_id)

    return (
        label_id_to_code,
        label_id_to_class,
        valid_label_ids,
        ambiguous_label_ids,
    )

print("Annotation-map function defined.")


In [ ]:
# Cell 8 — Load one preprocessed patient
def load_preprocessed_patient(patient_id):
    ct_path = PREPROCESSED_IMAGE_DIR / f"{patient_id}-image.npy"
    label_path = PREPROCESSED_LABEL_DIR / f"{patient_id}-label.npy"

    if not ct_path.exists():
        raise FileNotFoundError(f"CT file not found: {ct_path}")
    if not label_path.exists():
        raise FileNotFoundError(f"Label file not found: {label_path}")

    ct = np.load(ct_path).astype(np.float32, copy=False)
    label = np.load(label_path).astype(np.int16, copy=False)

    if ct.shape != label.shape:
        raise ValueError(f"CT/label mismatch: {ct.shape} vs {label.shape}")
    if ct.ndim != 3:
        raise ValueError(f"Expected 3D volume, got {ct.shape}")

    return ct, label

print("Patient-loading function defined.")


In [ ]:
# Cell 9 — Generate complete patch starting coordinates
def generate_patch_starts(volume_shape, patch_size, stride):
    starts = []

    for z in range(0, volume_shape[0] - patch_size[0] + 1, stride[0]):
        for y in range(0, volume_shape[1] - patch_size[1] + 1, stride[1]):
            for x in range(0, volume_shape[2] - patch_size[2] + 1, stride[2]):
                starts.append((z, y, x))

    return starts

print("Patch-coordinate function defined.")


In [ ]:
# Cell 10 — Convert annotation IDs into valid and ambiguous masks
def build_target_masks(label, label_id_to_code):
    valid_mask = np.zeros(label.shape, dtype=np.uint8)
    ambiguous_mask = np.zeros(label.shape, dtype=np.uint8)

    for label_id, label_code in label_id_to_code.items():
        if label_code in (1, 2, 3, 4):
            valid_mask[label == label_id] = 1
        elif label_code == AMBIGUOUS_LABEL_CODE:
            ambiguous_mask[label == label_id] = 1

    return valid_mask, ambiguous_mask

print("Target-mask function defined.")


In [ ]:
# Cell 11 — Determine a single classification target for a patch
def determine_patch_class(label_patch, label_id_to_code, min_fracture_voxels=1):
    class_counts = Counter()
    has_ambiguous = False

    for label_id in np.unique(label_patch):
        label_id = int(label_id)

        if label_id == 0:
            continue

        label_code = label_id_to_code.get(label_id)

        if label_code is None:
            has_ambiguous = True
            continue

        if label_code == AMBIGUOUS_LABEL_CODE:
            has_ambiguous = True
        elif label_code in (1, 2, 3, 4):
            class_id = LABEL_CODE_TO_CLASS[label_code]
            class_counts[class_id] += int(np.sum(label_patch == label_id))

    if has_ambiguous:
        return None, "ambiguous"

    total_valid_voxels = sum(class_counts.values())

    if total_valid_voxels < min_fracture_voxels:
        return 0, "background"

    max_count = max(class_counts.values())
    dominant_classes = [
        class_id for class_id, count in class_counts.items()
        if count == max_count
    ]

    if len(dominant_classes) != 1:
        return None, "mixed_class_tie"

    return dominant_classes[0], "positive"

print("Patch-classification function defined.")


## Patch extraction rules

Each candidate patch is assigned to one of three categories:

1. **Positive** — valid fracture voxels with a unique dominant fracture class.
2. **Clean negative** — no valid fracture voxels and no ambiguous annotations.
3. **Excluded** — ambiguous annotations or a tie between multiple valid fracture classes.

This prevents ambiguous labels from contaminating the segmentation or classification targets.


In [ ]:
# Cell 12 — Extract patches from one patient
def extract_patient_patches(
    patient_id,
    ct,
    label,
    patch_size=PATCH_SIZE,
    stride=STRIDE,
    negative_to_positive_ratio=NEGATIVE_TO_POSITIVE_RATIO,
    min_fracture_voxels=MIN_FRACTURE_VOXELS,
    seed=SEED,
):
    (
        label_id_to_code,
        label_id_to_class,
        valid_label_ids,
        ambiguous_label_ids,
    ) = get_patient_annotation_maps(patient_id)

    valid_mask, ambiguous_mask = build_target_masks(
        label, label_id_to_code
    )

    patch_starts = generate_patch_starts(
        ct.shape, patch_size, stride
    )

    positive_records = []
    negative_records = []
    excluded_counts = Counter()

    for start in patch_starts:
        z, y, x = start
        z2, y2, x2 = z + patch_size[0], y + patch_size[1], x + patch_size[2]

        ct_patch = ct[z:z2, y:y2, x:x2]
        label_patch = label[z:z2, y:y2, x:x2]
        valid_patch = valid_mask[z:z2, y:y2, x:x2]
        ambiguous_patch = ambiguous_mask[z:z2, y:y2, x:x2]

        class_id, reason = determine_patch_class(
            label_patch,
            label_id_to_code,
            min_fracture_voxels,
        )

        if reason == "ambiguous":
            excluded_counts["ambiguous"] += 1
            continue

        if reason == "mixed_class_tie":
            excluded_counts["mixed_class_tie"] += 1
            continue

        # Defensive check: an unexpected ambiguous voxel must never be selected.
        if np.any(ambiguous_patch):
            excluded_counts["ambiguous"] += 1
            continue

        if class_id == 0:
            if np.any(valid_patch):
                raise RuntimeError(
                    "Internal error: background patch contains valid fracture voxels."
                )

            negative_records.append({
                "ct": ct_patch,
                "seg": valid_patch,
                "class_id": 0,
                "start_z": z,
                "start_y": y,
                "start_x": x,
            })

        elif class_id in (1, 2, 3, 4):
            if not np.any(valid_patch):
                raise RuntimeError(
                    "Internal error: positive patch has no valid fracture voxels."
                )

            positive_records.append({
                "ct": ct_patch,
                "seg": valid_patch,
                "class_id": class_id,
                "start_z": z,
                "start_y": y,
                "start_x": x,
            })

        else:
            raise RuntimeError(f"Unexpected class ID: {class_id}")

    rng = random.Random(seed)
    rng.shuffle(positive_records)
    rng.shuffle(negative_records)

    max_negative = int(
        len(positive_records) * negative_to_positive_ratio
    )
    selected_negative = negative_records[:max_negative]

    selected_records = positive_records + selected_negative
    rng.shuffle(selected_records)

    if selected_records:
        X = np.stack(
            [record["ct"][..., np.newaxis] for record in selected_records],
            axis=0,
        ).astype(np.float32)

        Y_seg = np.stack(
            [record["seg"][..., np.newaxis] for record in selected_records],
            axis=0,
        ).astype(np.uint8)

        Y_class = np.asarray(
            [record["class_id"] for record in selected_records],
            dtype=np.int64,
        )

        manifest = pd.DataFrame([
            {
                "patient_id": patient_id,
                "start_z": record["start_z"],
                "start_y": record["start_y"],
                "start_x": record["start_x"],
                "classification": record["class_id"],
                "classification_name": CLASS_NAMES[record["class_id"]],
                "fracture_voxels": int(record["seg"].sum()),
            }
            for record in selected_records
        ])
    else:
        X = np.empty((0, *patch_size, 1), dtype=np.float32)
        Y_seg = np.empty((0, *patch_size, 1), dtype=np.uint8)
        Y_class = np.empty((0,), dtype=np.int64)
        manifest = pd.DataFrame()

    statistics = {
        "patient_id": patient_id,
        "candidate_patches": len(patch_starts),
        "positive_candidates": len(positive_records),
        "negative_candidates": len(negative_records),
        "selected_positive": len(positive_records),
        "selected_negative": len(selected_negative),
        "excluded_ambiguous": excluded_counts["ambiguous"],
        "excluded_mixed_class_tie": excluded_counts["mixed_class_tie"],
        "total_selected": len(selected_records),
    }

    return X, Y_seg, Y_class, manifest, statistics

print("Patient patch-extraction function defined.")


In [ ]:
# Cell 13 — Verify RibFrac128 annotation mapping
TEST_PATIENT_ID = "RibFrac128"

(
    label_id_to_code,
    label_id_to_class,
    valid_label_ids,
    ambiguous_label_ids,
) = get_patient_annotation_maps(TEST_PATIENT_ID)

print("=" * 70)
print(f"ANNOTATION MAPPING — {TEST_PATIENT_ID}")
print("=" * 70)

for label_id in sorted(label_id_to_code):
    print(f"label_id {label_id} -> label_code {label_id_to_code[label_id]}")

print("
Valid fracture annotation IDs:", sorted(valid_label_ids))
print("Ambiguous annotation IDs:", sorted(ambiguous_label_ids))

print("
Valid label_id -> classification class:")
for label_id in sorted(label_id_to_class):
    class_id = label_id_to_class[label_id]
    print(f"label_id {label_id} -> class {class_id} ({CLASS_NAMES[class_id]})")


In [ ]:
# Cell 14 — Load RibFrac128
ct, label = load_preprocessed_patient(TEST_PATIENT_ID)

print("=" * 70)
print(f"LOADED {TEST_PATIENT_ID}")
print("=" * 70)
print("CT shape:", ct.shape)
print("CT dtype:", ct.dtype)
print("CT range:", float(ct.min()), "to", float(ct.max()))
print("
Label shape:", label.shape)
print("Label dtype:", label.dtype)
print("Label IDs:", np.unique(label))

assert ct.shape == label.shape
assert ct.ndim == 3

print("
Patient volume verification: PASS")


In [ ]:
# Cell 15 — Verify RibFrac128 valid and ambiguous masks
valid_mask, ambiguous_mask = build_target_masks(
    label, label_id_to_code
)

print("=" * 70)
print("RIBFRAC128 TARGET MASK VERIFICATION")
print("=" * 70)
print("Valid fracture voxels:", int(valid_mask.sum()))
print("Ambiguous voxels:", int(ambiguous_mask.sum()))

expected_valid_voxels = int((label == 7).sum())

assert int(valid_mask.sum()) == expected_valid_voxels
assert not np.any((valid_mask == 1) & (ambiguous_mask == 1))

print("
Target-mask verification: PASS")


In [ ]:
# Cell 16 — Extract RibFrac128 patches
(
    X_patient,
    Y_seg_patient,
    Y_class_patient,
    manifest_patient,
    stats_patient,
) = extract_patient_patches(
    patient_id=TEST_PATIENT_ID,
    ct=ct,
    label=label,
    patch_size=PATCH_SIZE,
    stride=STRIDE,
    negative_to_positive_ratio=NEGATIVE_TO_POSITIVE_RATIO,
    min_fracture_voxels=MIN_FRACTURE_VOXELS,
    seed=SEED,
)

print("=" * 70)
print(f"PATCH EXTRACTION RESULTS — {TEST_PATIENT_ID}")
print("=" * 70)

for key, value in stats_patient.items():
    print(f"{key:<30}: {value}")

print("
X:", X_patient.shape, X_patient.dtype)
print("Y_seg:", Y_seg_patient.shape, Y_seg_patient.dtype)
print("Y_class:", Y_class_patient.shape, Y_class_patient.dtype)


In [ ]:
# Cell 17 — Validate patch shapes and targets
print("=" * 70)
print("VALIDATING EXTRACTED PATCHES")
print("=" * 70)

assert X_patient.ndim == 5
assert X_patient.shape[1:] == (*PATCH_SIZE, 1)

assert Y_seg_patient.ndim == 5
assert Y_seg_patient.shape[1:] == (*PATCH_SIZE, 1)

assert Y_class_patient.ndim == 1

assert X_patient.shape[0] == Y_seg_patient.shape[0] == Y_class_patient.shape[0]

assert float(X_patient.min()) >= 0.0
assert float(X_patient.max()) <= 1.0

assert set(np.unique(Y_seg_patient)).issubset({0, 1})
assert set(np.unique(Y_class_patient)).issubset({0, 1, 2, 3, 4})

positive_indices = np.where(Y_class_patient > 0)[0]
negative_indices = np.where(Y_class_patient == 0)[0]

for idx in positive_indices:
    assert Y_seg_patient[idx].sum() >= MIN_FRACTURE_VOXELS

for idx in negative_indices:
    assert Y_seg_patient[idx].sum() == 0

print("X shape:", X_patient.shape)
print("Y_seg shape:", Y_seg_patient.shape)
print("Y_class shape:", Y_class_patient.shape)
print("Segmentation values:", np.unique(Y_seg_patient))
print("Classification values:", np.unique(Y_class_patient))
print("Positive patches:", len(positive_indices))
print("Negative patches:", len(negative_indices))
print("
Patch target validation: PASS")


In [ ]:
# Cell 18 — Verify ambiguous annotations were excluded
ambiguous_patch_count = 0

for _, row in manifest_patient.iterrows():
    z = int(row["start_z"])
    y = int(row["start_y"])
    x = int(row["start_x"])

    label_patch = label[
        z:z + PATCH_SIZE[0],
        y:y + PATCH_SIZE[1],
        x:x + PATCH_SIZE[2],
    ]

    if any(
        int(label_id) in ambiguous_label_ids
        for label_id in np.unique(label_patch)
    ):
        ambiguous_patch_count += 1

print("Selected patches containing ambiguous annotations:", ambiguous_patch_count)

assert ambiguous_patch_count == 0
print("Ambiguous-annotation exclusion: PASS")


In [ ]:
# Cell 19 — Inspect classification distribution
class_counts = pd.Series(Y_class_patient).value_counts().sort_index()

classification_summary = pd.DataFrame({
    "class_id": class_counts.index.astype(int),
    "class_name": [
        CLASS_NAMES[int(class_id)] for class_id in class_counts.index
    ],
    "patch_count": class_counts.values,
})

display(classification_summary)


In [ ]:
# Cell 20 — Inspect segmentation statistics
fracture_voxels_per_patch = Y_seg_patient.sum(axis=(1, 2, 3, 4))

print("Total patches:", len(Y_seg_patient))
print("Positive patches:", int(np.sum(fracture_voxels_per_patch > 0)))
print("Negative patches:", int(np.sum(fracture_voxels_per_patch == 0)))

positive_voxel_counts = fracture_voxels_per_patch[
    fracture_voxels_per_patch > 0
]

if len(positive_voxel_counts):
    print("Minimum positive fracture voxels:", int(positive_voxel_counts.min()))
    print("Maximum positive fracture voxels:", int(positive_voxel_counts.max()))


In [ ]:
# Cell 21 — Save RibFrac128 patch dataset
TEST_PATCH_DIR = PATCH_ROOT / TEST_PATIENT_ID
TEST_MANIFEST_PATH = PATCH_MANIFEST_ROOT / f"{TEST_PATIENT_ID}_manifest.csv"

TEST_PATCH_DIR.mkdir(parents=True, exist_ok=True)
PATCH_MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)

X_PATH = TEST_PATCH_DIR / "X.npy"
Y_SEG_PATH = TEST_PATCH_DIR / "Y_seg.npy"
Y_CLASS_PATH = TEST_PATCH_DIR / "Y_class.npy"

np.save(X_PATH, X_patient)
np.save(Y_SEG_PATH, Y_seg_patient)
np.save(Y_CLASS_PATH, Y_class_patient)
manifest_patient.to_csv(TEST_MANIFEST_PATH, index=False)

print("=" * 70)
print("RIBFRAC128 PATCH DATA SAVED")
print("=" * 70)
print("X:", X_PATH)
print("Y_seg:", Y_SEG_PATH)
print("Y_class:", Y_CLASS_PATH)
print("Manifest:", TEST_MANIFEST_PATH)


In [ ]:
# Cell 22 — Save extraction configuration
CONFIG_PATH = PATCH_ROOT / "patch_extraction_config.json"

config = {
    "seed": SEED,
    "patch_size": list(PATCH_SIZE),
    "stride": list(STRIDE),
    "negative_to_positive_ratio": NEGATIVE_TO_POSITIVE_RATIO,
    "minimum_fracture_voxels": MIN_FRACTURE_VOXELS,
    "classification_classes": CLASS_NAMES,
    "ambiguous_label_code": AMBIGUOUS_LABEL_CODE,
    "segmentation_definition": {
        "0": "background",
        "1": "valid fracture",
    },
    "classification_definition": CLASS_NAMES,
    "dataset_root": str(DATASET_ROOT),
    "preprocessed_root": str(PREPROCESSED_ROOT),
    "patient_split_path": str(SPLIT_PATH),
    "annotation_metadata_path": str(INFO_PATH),
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("Configuration saved:", CONFIG_PATH)


In [ ]:
# Cell 23 — Reload saved RibFrac128 patches
saved_X = np.load(X_PATH, mmap_mode="r")
saved_Y_seg = np.load(Y_SEG_PATH, mmap_mode="r")
saved_Y_class = np.load(Y_CLASS_PATH, mmap_mode="r")
saved_manifest = pd.read_csv(TEST_MANIFEST_PATH)

assert saved_X.shape == X_patient.shape
assert saved_Y_seg.shape == Y_seg_patient.shape
assert saved_Y_class.shape == Y_class_patient.shape
assert len(saved_manifest) == X_patient.shape[0]

print("=" * 70)
print("SAVED PATCH RELOAD VALIDATION")
print("=" * 70)
print("Reloaded X:", saved_X.shape)
print("Reloaded Y_seg:", saved_Y_seg.shape)
print("Reloaded Y_class:", saved_Y_class.shape)
print("Reloaded manifest rows:", len(saved_manifest))
print("
Saved patch reload validation: PASS")


In [ ]:
# Cell 24 — Verify manifest integrity
required_manifest_columns = {
    "patient_id",
    "start_z",
    "start_y",
    "start_x",
    "classification",
    "classification_name",
    "fracture_voxels",
}

missing_manifest_columns = required_manifest_columns - set(saved_manifest.columns)
assert not missing_manifest_columns, f"Missing manifest columns: {missing_manifest_columns}"

assert set(saved_manifest["patient_id"]) == {TEST_PATIENT_ID}
assert set(saved_manifest["classification"].astype(int).unique()).issubset({0, 1, 2, 3, 4})

print("Manifest integrity: PASS")


In [ ]:
# Cell 25 — Report generated files
print("=" * 70)
print("GENERATED PATCH DATASET")
print("=" * 70)

for path in sorted(PATCH_ROOT.rglob("*")):
    if path.is_file():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"{path} | {size_mb:.3f} MB")


In [ ]:
# Cell 26 — Release RibFrac128 arrays from memory
del X_patient, Y_seg_patient, Y_class_patient, manifest_patient
del saved_X, saved_Y_seg, saved_Y_class, saved_manifest
del ct, label
gc.collect()

print("Temporary RibFrac128 arrays released.")


## Full split-wise extraction

The full development dataset is deliberately disabled during pipeline validation.

Current development split:

- Training: 192 patients
- Validation: 48 patients
- Test: 60 patients

After Notebooks 03–07 have been validated:

- set `PROCESS_TRAINING_SET = True` for the 192-patient training split;
- set `PROCESS_VALIDATION_SET = True` for the 48-patient validation split;
- keep `PROCESS_TEST_SET = False` unless the selected development test cases have appropriate ground truth and are intentionally used as a held-out evaluation set.

No patch-level random split is performed.


In [ ]:
# Cell 27 — Full extraction switches
PROCESS_TRAINING_SET = False
PROCESS_VALIDATION_SET = False
PROCESS_TEST_SET = False

print("Training extraction:", PROCESS_TRAINING_SET)
print("Validation extraction:", PROCESS_VALIDATION_SET)
print("Test extraction:", PROCESS_TEST_SET)


In [ ]:
# Cell 28 — Process and save one patient
def process_and_save_patient(patient_id, split_name):
    print("=" * 70)
    print(f"Processing {patient_id} | split = {split_name}")
    print("=" * 70)

    ct, label = load_preprocessed_patient(patient_id)

    X, Y_seg, Y_class, manifest, statistics = extract_patient_patches(
        patient_id=patient_id,
        ct=ct,
        label=label,
        patch_size=PATCH_SIZE,
        stride=STRIDE,
        negative_to_positive_ratio=NEGATIVE_TO_POSITIVE_RATIO,
        min_fracture_voxels=MIN_FRACTURE_VOXELS,
        seed=SEED,
    )

    patient_patch_dir = PATCH_ROOT / split_name / patient_id
    patient_manifest_dir = PATCH_MANIFEST_ROOT / split_name

    patient_patch_dir.mkdir(parents=True, exist_ok=True)
    patient_manifest_dir.mkdir(parents=True, exist_ok=True)

    np.save(patient_patch_dir / "X.npy", X)
    np.save(patient_patch_dir / "Y_seg.npy", Y_seg)
    np.save(patient_patch_dir / "Y_class.npy", Y_class)

    manifest_path = patient_manifest_dir / f"{patient_id}_manifest.csv"
    manifest.to_csv(manifest_path, index=False)

    del ct, label, X, Y_seg, Y_class, manifest
    gc.collect()

    print("Saved patches:", statistics["total_selected"])
    print("Statistics:", statistics)

    return statistics

print("Full patient-processing function defined.")


In [ ]:
# Cell 29 — Process a complete predefined split
def process_split(patient_ids, split_name):
    split_statistics = []

    print("=" * 70)
    print(f"STARTING SPLIT EXTRACTION: {split_name}")
    print(f"Number of patients: {len(patient_ids)}")
    print("=" * 70)

    for index, patient_id in enumerate(patient_ids, start=1):
        print(f"\n[{index}/{len(patient_ids)}] {patient_id}")

        try:
            stats = process_and_save_patient(patient_id, split_name)
            split_statistics.append(stats)
        except Exception as exc:
            print(f"ERROR processing {patient_id}: {exc}")
            split_statistics.append({
                "patient_id": patient_id,
                "error": str(exc),
            })

    statistics_df = pd.DataFrame(split_statistics)

    summary_path = (
        PATCH_MANIFEST_ROOT / split_name / "split_extraction_summary.csv"
    )
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    statistics_df.to_csv(summary_path, index=False)

    print("=" * 70)
    print(f"COMPLETED SPLIT: {split_name}")
    print("Summary:", summary_path)
    print("=" * 70)

    return statistics_df


split_patient_lists = {
    "train": sorted(train_ids),
    "validation": sorted(val_ids),
    "test": sorted(test_ids),
}

if PROCESS_TRAINING_SET:
    train_statistics = process_split(split_patient_lists["train"], "train")

if PROCESS_VALIDATION_SET:
    validation_statistics = process_split(split_patient_lists["validation"], "validation")

if PROCESS_TEST_SET:
    test_statistics = process_split(split_patient_lists["test"], "test")

if not (PROCESS_TRAINING_SET or PROCESS_VALIDATION_SET or PROCESS_TEST_SET):
    print("Full split extraction is disabled. Only the RibFrac128 development test is processed.")


# Notebook 03 completion criteria

Notebook 03 is validated when RibFrac128 demonstrates:

1. Correct persistent-data access.
2. Correct reuse of the patient-level split.
3. Correct annotation-ID to `label_code` mapping.
4. Correct exclusion of ambiguous (`-1`) annotations.
5. Correct 64×64×64 CT patch extraction.
6. Binary segmentation targets `{0,1}`.
7. Five-class classification targets `{0,1,2,3,4}`.
8. No ambiguous voxels in selected patches.
9. Positive patches contain valid fracture voxels.
10. Negative patches contain no valid fracture voxels.
11. CT and target arrays have consistent patch counts.
12. Saved patch files reload successfully.
13. No patch-level train/validation split is performed.

## RibFrac128 development case

For RibFrac128:

- annotation ID `7` → `label_code = 3` → **buckle fracture**;
- annotation IDs `1–6` and `8` → `label_code = -1` → **ambiguous/ignore**.

Therefore, the RibFrac128 test should produce buckle-class positive patches only. It is an end-to-end pipeline test and must not be interpreted as representative model-performance evidence.

## Output structure

```text
/kaggle/working/patch_dataset/
├── patch_extraction_config.json
├── RibFrac128/
│   ├── X.npy
│   ├── Y_seg.npy
│   └── Y_class.npy
├── train/
│   └── <patient_id>/
│       ├── X.npy
│       ├── Y_seg.npy
│       └── Y_class.npy
├── validation/
│   └── <patient_id>/
│       ├── X.npy
│       ├── Y_seg.npy
│       └── Y_class.npy
└── manifests/
    ├── RibFrac128_manifest.csv
    ├── train/
    └── validation/
```

Notebook 04 should consume these outputs using a **five-class softmax classification head** and a binary segmentation head.
